# 08 - Introducción a Agentes de IA

## Curso de LLMs y Aplicaciones de IA

**Duración estimada:** 2-2.5 horas

---

## Índice

1. [¿Qué son los Agentes?](#intro)
2. [De RAG a Agentes](#rag2agents)
3. [Componentes de un Agente](#componentes)
4. [Planificación: Chain of Thought y ReAct](#planificacion)
5. [Primer Agente simple](#primer)
6. [Ejercicios prácticos](#ejercicios)

---

## Objetivos de aprendizaje

Al finalizar este notebook, serás capaz de:
- Entender qué diferencia a un agente de un LLM simple
- Conocer los componentes clave: planificación, memoria, herramientas
- Implementar técnicas de razonamiento como ReAct
- Crear un agente básico que use herramientas

<a name="intro"></a>
## 1. ¿Qué son los Agentes?

Un **agente de IA** es un sistema que puede:
1. **Percibir** su entorno (recibir inputs)
2. **Razonar** sobre qué hacer (planificación)
3. **Actuar** para lograr objetivos (ejecutar herramientas)
4. **Aprender** de la experiencia (memoria)

### LLM vs Agente

| LLM Simple | Agente |
|------------|--------|
| Responde preguntas | Ejecuta tareas |
| Sin estado | Con memoria |
| Solo texto | Usa herramientas |
| Un paso | Múltiples pasos |
| Pasivo | Proactivo |

<a name="rag2agents"></a>
## 2. De RAG a Agentes

RAG fue un primer paso para dar a los LLMs acceso a información externa. Los agentes expanden esta idea:

```
┌─────────────────────────────────────────────┐
│                    AGENTE                    │
├─────────────────────────────────────────────┤
│  ┌─────────────────┐                        │
│  │   PLANIFICACIÓN │ ← Descompone tareas    │
│  │   (Razonamiento)│   ReAct, CoT           │
│  └────────┬────────┘                        │
│           │                                 │
│  ┌────────▼────────┐                        │
│  │     MEMORIA     │ ← Corto y largo plazo  │
│  │  (Contexto)     │   Chat history, RAG    │
│  └────────┬────────┘                        │
│           │                                 │
│  ┌────────▼────────┐                        │
│  │   HERRAMIENTAS  │ ← Búsqueda, cálculo,   │
│  │    (Acciones)   │   APIs, código         │
│  └─────────────────┘                        │
└─────────────────────────────────────────────┘
```

In [4]:
# Install required libraries
!pip install -q langchain langchain-groq langchain-community

In [5]:
import os
from getpass import getpass
import warnings
warnings.filterwarnings('ignore')

if 'GROQ_API_KEY' not in os.environ:
    os.environ['GROQ_API_KEY'] = getpass("Introduce tu GROQ API Key: ")

print("Configuración completada ✓")

Configuración completada ✓


<a name="componentes"></a>
## 3. Componentes de un Agente

### 3.1 Planificación

El agente debe ser capaz de:
- **Descomponer** tareas complejas en subtareas
- **Razonar** sobre qué hacer a continuación
- **Auto-corregirse** si algo falla

### 3.2 Memoria

| Tipo | Descripción | Implementación |
|------|-------------|----------------|
| **Sensorial** | Input actual | El prompt |
| **Corto plazo** | Conversación actual | Chat history |
| **Largo plazo** | Conocimiento persistente | Vector store (RAG) |

### 3.3 Herramientas

Funciones que el agente puede llamar:
- Búsqueda web
- Calculadora
- Ejecución de código
- APIs externas
- Bases de datos

<a name="planificacion"></a>
## 4. Planificación: Chain of Thought y ReAct

### Chain of Thought (CoT)

Pedir al modelo que "piense paso a paso" mejora el razonamiento.

In [6]:
from langchain_groq import ChatGroq
from langchain_classic import hub

llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0)

# Chain of Thought example
cot_prompt = """Resuelve el siguiente problema paso a paso.

Problema: Una tienda ofrece 20% de descuento en todos los productos.
Si un artículo cuesta 80€, y además hay un cupón de 5€ de descuento adicional,
¿cuánto pagaría el cliente?

Piensa paso a paso:"""

response = llm.invoke(cot_prompt)
print(response.content)

Para resolver este problema, seguiremos los pasos siguientes:

**Paso 1: Calcular el descuento del 20%**

* El artículo cuesta 80€.
* El descuento es del 20%, por lo que debemos calcular el 20% de 80€.
* 20% de 80€ = 0,20 x 80 = 16€.

**Paso 2: Restar el descuento del precio original**

* El precio original es 80€.
* El descuento es de 16€.
* El precio después del descuento es: 80 - 16 = 64€.

**Paso 3: Aplicar el cupón de descuento adicional**

* El cliente tiene un cupón de 5€ de descuento adicional.
* El precio después del descuento es 64€.
* El precio final después de aplicar el cupón es: 64 - 5 = 59€.

**Conclusión**

El cliente pagaría 59€ por el artículo después de aplicar el descuento del 20% y el cupón de 5€ de descuento adicional.


### ReAct: Reasoning + Acting

**ReAct** combina razonamiento con acciones. El patrón es:

```
Thought: [Razonamiento sobre qué hacer]
Action: [Herramienta a usar]
Action Input: [Parámetros para la herramienta]
Observation: [Resultado de la acción]
... (repetir hasta tener respuesta)
Final Answer: [Respuesta al usuario]
```

In [7]:
# ReAct prompt structure
react_prompt = """Responde la pregunta usando el formato ReAct.

Herramientas disponibles:
- calculator: Para operaciones matemáticas
- search: Para buscar información

Formato:
Thought: [Tu razonamiento]
Action: [Nombre de herramienta]
Action Input: [Input para la herramienta]
Observation: [Resultado - lo simularemos]
... (repetir si es necesario)
Final Answer: [Tu respuesta final]

Pregunta: ¿Cuántos segundos hay en 3 días, 4 horas y 30 minutos?

Comienza:"""

response = llm.invoke(react_prompt)
print(response.content)

Thought: Para calcular el número total de segundos en 3 días, 4 horas y 30 minutos, debemos convertir cada unidad de tiempo a segundos. Primero, calcularé el número de segundos en 3 días.

Action: calculator
Action Input: 3 * 24 * 60 * 60
Observation: 259200

Thought: Ahora, calcularé el número de segundos en 4 horas.

Action: calculator
Action Input: 4 * 60 * 60
Observation: 14400

Thought: Luego, calcularé el número de segundos en 30 minutos.

Action: calculator
Action Input: 30 * 60
Observation: 1800

Thought: Finalmente, sumaré los segundos de los días, horas y minutos para obtener el total de segundos.

Action: calculator
Action Input: 259200 + 14400 + 1800
Observation: 278400

Final Answer: 278400


<a name="primer"></a>
## 5. Primer Agente Simple

Vamos a crear un agente básico con herramientas personalizadas.

In [8]:
from langchain.tools import tool

# Define custom tools
@tool
def calculator(expression: str) -> str:
    """Evalúa una expresión matemática. Ejemplo: '2 + 2' o '10 * 5'"""
    try:
        # Safe evaluation
        allowed = set('0123456789+-*/(). ')
        if not all(c in allowed for c in expression):
            return "Error: Expresión inválida"
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {e}"

@tool
def get_current_time() -> str:
    """Obtiene la fecha y hora actual."""
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

@tool
def string_length(text: str) -> str:
    """Calcula la longitud de un texto."""
    return str(len(text))

# List tools
tools = [calculator, get_current_time, string_length]

print("Herramientas disponibles:")
for t in tools:
    print(f"  - {t.name}: {t.description}")

Herramientas disponibles:
  - calculator: Evalúa una expresión matemática. Ejemplo: '2 + 2' o '10 * 5'
  - get_current_time: Obtiene la fecha y hora actual.
  - string_length: Calcula la longitud de un texto.


In [9]:
# Test tools directly
print("Test calculator:", calculator.invoke("15 * 4 + 10"))
print("Test time:", get_current_time.invoke(""))
print("Test length:", string_length.invoke("Hola mundo"))

Test calculator: 70
Test time: 2026-06-10 19:21:30
Test length: 10


In [16]:
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

# Create agent prompt
agent_prompt = ChatPromptTemplate.from_messages([
    ("system", """Eres un asistente útil con acceso a herramientas.
Usa las herramientas cuando sea necesario para responder preguntas.
Siempre muestra tu razonamiento."""),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

# Create the agent
agent = create_tool_calling_agent(llm, tools, agent_prompt)

# Create executor
agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=False  # Show reasoning
)

print("Agente creado ✓")

Agente creado ✓


In [17]:
# Test the agent
result = agent_executor.invoke({
    "input": "¿Cuánto es 25 multiplicado por 4, más 150?",
    "chat_history": []
})

print("\n" + "="*50)
print(f"Respuesta final: {result['output']}")


Respuesta final: Primero, multiplicamos 25 por 4, lo que da como resultado 100. Luego, sumamos 150 a ese resultado, lo que da como resultado 250. La respuesta final es 250.


In [18]:
# Test with multiple tools
result = agent_executor.invoke({
    "input": "¿Qué hora es y cuántos caracteres tiene la frase 'Inteligencia Artificial'?",
    "chat_history": []
})

print("\n" + "="*50)
print(f"Respuesta final: {result['output']}")


Respuesta final: La hora actual es 19:30:54 y la frase 'Inteligencia Artificial' tiene 23 caracteres.


<a name="ejercicios"></a>
## 6. Ejercicios Prácticos

### Ejercicio 1: Crear una herramienta personalizada

In [13]:
# Exercise 1: Create a custom tool
# Ideas:
# - Temperature converter (Celsius to Fahrenheit)
# - Random number generator
# - Word counter

@tool
def celsius_to_fahrenheit(celsius: str) -> str:
    """Convierte temperatura de Celsius a Fahrenheit. Input: número en Celsius."""
    try:
        c = float(celsius)
        f = (c * 9/5) + 32
        return f"{c}°C = {f}°F"
    except:
        return "Error: Proporciona un número válido"

# Test your tool
print(celsius_to_fahrenheit.invoke("25"))

25.0°C = 77.0°F


In [23]:
from langchain.tools import tool
import requests

@tool
def temperatura_actual(ciudad: str) -> str:
    """
    Devuelve la temperatura actual en una ciudad.
    Input: nombre de una ciudad, por ejemplo 'Málaga', 'Madrid' o 'Sevilla'.
    """
    try:
        # 1. Convertir ciudad a coordenadas
        geo_url = "https://geocoding-api.open-meteo.com/v1/search"
        geo_params = {
            "name": ciudad,
            "count": 1,
            "language": "es",
            "format": "json"
        }

        geo_response = requests.get(geo_url, params=geo_params, timeout=10)
        geo_response.raise_for_status()
        geo_data = geo_response.json()

        if "results" not in geo_data or len(geo_data["results"]) == 0:
            return f"No he encontrado la ciudad: {ciudad}"

        lugar = geo_data["results"][0]
        lat = lugar["latitude"]
        lon = lugar["longitude"]
        nombre = lugar["name"]
        pais = lugar.get("country", "")

        # 2. Consultar la temperatura actual
        weather_url = "https://api.open-meteo.com/v1/forecast"
        weather_params = {
            "latitude": lat,
            "longitude": lon,
            "current": "temperature_2m",
            "timezone": "auto"
        }

        weather_response = requests.get(weather_url, params=weather_params, timeout=10)
        weather_response.raise_for_status()
        weather_data = weather_response.json()

        temperatura = weather_data["current"]["temperature_2m"]
        unidad = weather_data["current_units"]["temperature_2m"]

        return f"Ahora mismo en {nombre}, {pais}, hace {temperatura}{unidad}."

    except Exception as e:
        return f"Error al consultar la temperatura: {e}"

tools = [
    celsius_to_fahrenheit,
    temperatura_actual
]        

# Create the agent
agent = create_tool_calling_agent(llm, tools, agent_prompt)
 
# Create executor
agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True  # Show reasoning
)

In [24]:
# Test the agent
result = agent_executor.invoke({
    "input": "Voy a ir unos dias a Tokio, ¿Que ropa deberia llevar?",
    "chat_history": []
})



> Entering new AgentExecutor chain...

Invoking: `temperatura_actual` with `{'ciudad': 'Tokio'}`
responded: Para determinar la ropa adecuada para llevar a Tokio, necesitamos saber la temperatura actual en la ciudad. 



Ahora mismo en Tokio, Japón, hace 17.5°C.
Invoking: `celsius_to_fahrenheit` with `{'celsius': '17.5'}`


17.5°C = 63.5°FCon una temperatura de 17.5°C (63.5°F) en Tokio, te recomiendo llevar ropa ligera y cómoda, como camisetas, pantalones cortos y zapatos cómodos. También es una buena idea llevar una chaqueta o un suéter ligero para las noches más frescas. No olvides empacar ropa adecuada para visitar templos o lugares culturales, como ropa modesta y cómoda. ¡Disfruta tu viaje a Tokio!

> Finished chain.


In [21]:
print(temperatura_actual.invoke("Tokio"))

Ahora mismo en Tokio, Japón, hace 17.5°C.


### Ejercicio 2: Agente con múltiples herramientas

In [14]:
# Exercise 2: Create an agent with your new tools
# Add the celsius_to_fahrenheit tool and test the agent

# extended_tools = tools + [celsius_to_fahrenheit]
# Create new agent with extended tools
# Test with: "Si la temperatura es 30 grados Celsius, ¿cuánto es en Fahrenheit?"

## Resumen

En este notebook hemos aprendido:

1. **Agentes**: LLMs que pueden razonar, recordar y actuar
2. **Componentes**: Planificación, memoria, herramientas
3. **Chain of Thought**: Razonamiento paso a paso
4. **ReAct**: Combinación de pensamiento y acción
5. **Herramientas**: Funciones que el agente puede llamar

### Arquitectura de un Agente

```
Input → [Planificación] → [Selección de herramienta] → [Ejecución] → [Observación] → [Repetir o Responder]
```

En el siguiente notebook profundizaremos en **Agentes con LangChain**, incluyendo herramientas de búsqueda y RAG.

---

## Referencias

- [LangChain Agents](https://python.langchain.com/docs/modules/agents/)
- [ReAct Paper](https://arxiv.org/abs/2210.03629)
- [Lilian Weng's Agent Blog](https://lilianweng.github.io/posts/2023-06-23-agent/)

In [15]:
import session_info
session_info.show(html = False)

-----
ipykernel           6.31.0
langchain_classic   1.0.7
langchain_core      1.4.0
langchain_groq      1.1.2
pandas              2.3.3
session_info        v1.0.1
-----
IPython             9.7.0
jupyter_client      8.6.3
jupyter_core        5.8.1
jupyterlab          4.4.7
notebook            7.4.5
-----
Python 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]
Windows-10-10.0.19045-SP0
-----
Session information updated at 2026-06-10 19:21
